# Kaggle setup and hostile audit

Run this once in each Kaggle account before launching a training notebook.
Attach the same Kaggle Dataset containing `train_set.npz` and `teacher_logp.npy` to every kernel.
The exact MATE and puzzle evaluation files must remain separate from the training dataset.

In [ ]:
from pathlib import Path
import os, subprocess, sys, json, platform

REPO = Path('/kaggle/working/chess-slm-benchmark')
DATA = Path('/kaggle/input/chessbench-full')  # full training export
SL_REPO = Path('/kaggle/working/searchless_chess')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Vedang-P/chess-slm-benchmark.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
if not SL_REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/google-deepmind/searchless_chess.git', str(SL_REPO)], check=True)
os.chdir(REPO)
print('python:', sys.version)
print('platform:', platform.platform())
print('cuda:', subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())
print('repo:', subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip())

In [ ]:
%pip install -q numpy torch pandas huggingface_hub python-chess dm-haiku jaxtyping zstandard apache-beam grain
import numpy as np
train_path = DATA / 'train_set.npz'
teacher_path = DATA / 'teacher_logp.npy'
assert train_path.exists(), train_path
assert teacher_path.exists(), teacher_path
d = np.load(train_path)
teacher = np.load(teacher_path, mmap_mode='r')
print({k: (v.shape, str(v.dtype)) for k, v in d.items()})
print('teacher:', teacher.shape, teacher.dtype)
assert d['tokens'].shape[0] == teacher.shape[0]
assert d['tokens'].shape[1] == 77
norm = np.logaddexp.reduce(np.asarray(teacher[:1024]), axis=1)
print('teacher logsumexp max abs:', float(np.max(np.abs(norm))))
assert np.allclose(norm, 0, atol=2e-3), 'teacher is not normalized log-probability data'

## Launch policy

Run one notebook per GPU. Two kernels may run concurrently on one Kaggle account; do not place two training processes in one kernel because they will contend for one GPU and make throughput measurements meaningless.

Use unique `--hf-run` names such as `account1-baseline-5m-seed0` and `account1-gavn-3m-seed0`. Never let two jobs upload to the same HF prefix.